In [1]:
import os
import boto3
from sagemaker import get_execution_role
from pprint import pprint
import json
import time
import pandas as pd
import numpy as np
from tqdm import tqdm
import math
import datetime as dt

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-03-07 20:50:25.804100


### Constants

In [3]:
# name of step function
str_name = 'gen-xii-retro-scoring'
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')
# subtask
str_subtask = os.getcwd().split('/')[6]
print(f'Subtask: {str_subtask}')
int_n_requests_per_lambda = 100

Project: 20231010-gen-xii
Task: 08_retro_scoring
Subtask: 03_step_function


### Make ```rows``` column in ```df_requests.gzip```

In [4]:
%%time

# import data
str_filename = 'df_requests.gzip'
str_uri = f's3://{str_project}/{str_task}/01_get_requests_from_db/{str_filename}'
df = pd.read_parquet(str_uri)

# show
df

CPU times: user 40.2 s, sys: 19.3 s, total: 59.6 s
Wall time: 1min 29s


,bigAccountId,dtmFunded,strRequest
4713,5709029,2021-07-27,"{""request_id"":""589151"",""rows"":[{""row_id"":""5709..."
4718,5712734,2021-07-27,"{""request_id"":""589440"",""rows"":[{""row_id"":""5712..."
4729,5709088,2021-07-29,"{""request_id"":""589470"",""rows"":[{""row_id"":""5709..."
4742,5718748,2021-07-28,"{""request_id"":""589569"",""rows"":[{""row_id"":""5718..."
4755,5717887,2021-07-27,"{""request_id"":""589609"",""rows"":[{""row_id"":""5717..."
...,...,...,...
69710,7372352,2023-11-30,"{""request_id"":""7372352586908"",""rows"":[{""row_id..."
69711,7107327,2023-12-21,"{""request_id"":""7107327371257"",""rows"":[{""row_id..."
69712,7351212,2023-12-04,"{""request_id"":""7351212932709"",""rows"":[{""row_id..."
69713,7358983,2023-12-04,"{""request_id"":""7358983628454"",""rows"":[{""row_id..."


In [5]:
# keep only the most recent payload
df.drop_duplicates(
    subset=['bigAccountId'],
    keep='last', 
    inplace=True,
)
# show
df

,bigAccountId,dtmFunded,strRequest
4713,5709029,2021-07-27,"{""request_id"":""589151"",""rows"":[{""row_id"":""5709..."
4718,5712734,2021-07-27,"{""request_id"":""589440"",""rows"":[{""row_id"":""5712..."
4729,5709088,2021-07-29,"{""request_id"":""589470"",""rows"":[{""row_id"":""5709..."
4742,5718748,2021-07-28,"{""request_id"":""589569"",""rows"":[{""row_id"":""5718..."
4755,5717887,2021-07-27,"{""request_id"":""589609"",""rows"":[{""row_id"":""5717..."
...,...,...,...
69710,7372352,2023-11-30,"{""request_id"":""7372352586908"",""rows"":[{""row_id..."
69711,7107327,2023-12-21,"{""request_id"":""7107327371257"",""rows"":[{""row_id..."
69712,7351212,2023-12-04,"{""request_id"":""7351212932709"",""rows"":[{""row_id..."
69713,7358983,2023-12-04,"{""request_id"":""7358983628454"",""rows"":[{""row_id..."


In [6]:
# sort
df.sort_values(by='dtmFunded', ascending=True, inplace=True)

# show
df

,bigAccountId,dtmFunded,strRequest
43726,5514485,2021-01-25,"{""request_id"":""5514485988291"",""rows"":[{""row_id..."
37788,5517730,2021-01-27,"{""request_id"":""5517730110604"",""rows"":[{""row_id..."
44805,5515245,2021-01-27,"{""request_id"":""5515245687539"",""rows"":[{""row_id..."
44811,5518455,2021-01-28,"{""request_id"":""5518455163761"",""rows"":[{""row_id..."
41809,5514970,2021-01-28,"{""request_id"":""5514970553077"",""rows"":[{""row_id..."
...,...,...,...
67406,7321766,2024-02-08,"{""request_id"":""7321766280426"",""rows"":[{""row_id..."
66872,7306479,2024-02-09,"{""request_id"":""7306479330180"",""rows"":[{""row_id..."
62902,7359135,2024-02-09,"{""request_id"":""7359135706073"",""rows"":[{""row_id..."
67709,7351735,2024-02-28,"{""request_id"":""7351735580851"",""rows"":[{""row_id..."


In [7]:
# get min and max dates
dtm_min = df['dtmFunded'].min()
dtm_max = df['dtmFunded'].max()
print(f'Min date: {dtm_min.date()}; Max date: {dtm_max.date()}')

Min date: 2021-01-25; Max date: 2024-03-04


In [8]:
# get nrows
int_nrows = df.shape[0]

# divide by int_n_requests_per_lambda
int_n_lambdas = math.ceil(int_nrows / int_n_requests_per_lambda)
print(f'There will be {int_n_lambdas} lambdas')

There will be 585 lambdas


In [9]:
# create list to assign as new column
list_rows = list(np.tile(np.arange(1, int_n_lambdas+1), int_n_requests_per_lambda))
print(f'Length: {len(list_rows)}')
# make sure its the same length as df
list_rows = list_rows[:int_nrows]
print(f'Length: {len(list_rows)}')
# assign
df['rows'] = list_rows
# show
df

Length: 58500
Length: 58431


,bigAccountId,dtmFunded,strRequest,rows
43726,5514485,2021-01-25,"{""request_id"":""5514485988291"",""rows"":[{""row_id...",1
37788,5517730,2021-01-27,"{""request_id"":""5517730110604"",""rows"":[{""row_id...",2
44805,5515245,2021-01-27,"{""request_id"":""5515245687539"",""rows"":[{""row_id...",3
44811,5518455,2021-01-28,"{""request_id"":""5518455163761"",""rows"":[{""row_id...",4
41809,5514970,2021-01-28,"{""request_id"":""5514970553077"",""rows"":[{""row_id...",5
...,...,...,...,...
67406,7321766,2024-02-08,"{""request_id"":""7321766280426"",""rows"":[{""row_id...",512
66872,7306479,2024-02-09,"{""request_id"":""7306479330180"",""rows"":[{""row_id...",513
62902,7359135,2024-02-09,"{""request_id"":""7359135706073"",""rows"":[{""row_id...",514
67709,7351735,2024-02-28,"{""request_id"":""7351735580851"",""rows"":[{""row_id...",515


In [10]:
# %%time

# # import data
# str_filename = 'df_requests.gzip'
# str_uri = f's3://{str_project}/{str_task}/01_get_requests_from_db/{str_filename}'
# df = pd.read_parquet(str_uri)

# int_nrows = df.shape[0]
# while True:
#     int_n_lambdas = int_nrows / int_n_requests_per_lambda
#     # if int_n_lambdas becomes a whole number  then break
#     if int_n_lambdas % 1 == 0:
#         break
#     else:
#         int_n_requests_per_lambda += 1
# print(f'Number of lambdas: {int_n_lambdas}')
# print(f'Number of requests per lambda : {int_n_requests_per_lambda}')
# # create list to assign as new column
# list_rows = list(np.tile(np.arange(1, int_n_lambdas+1), int_n_requests_per_lambda))

# # assign
# df['rows'] = list_rows
# # make int
# df['rows'] = df['rows'].astype(int)

# # show
# df

### Make ```df_idx.csv```

In [11]:
list_rows = list(df['rows'].value_counts().index)
df_idx = pd.DataFrame({'row': list_rows})
df_idx.sort_values(by='row', ascending=True, inplace=True)

# save
str_filename = 'df_idx.csv'
str_uri = f's3://{str_project}/{str_task}/{str_subtask}/{str_filename}'
df_idx.to_csv(str_uri, index=False)

# show
df_idx

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:273: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,row
0,1
98,2
354,3
353,4
352,5
...,...
538,581
532,582
537,583
536,584


### Subset and save

In [12]:
%%time

for int_row in tqdm(df_idx['row']):
    # subset
    df_tmp = df[df['rows'] == int_row].copy()
    # save
    str_filename = f'df_rows_{int_row}.gzip'
    str_uri = f's3://{str_project}/{str_task}/{str_subtask}/rows/{str_filename}'
    df_tmp.to_parquet(str_uri, compression='gzip')

100%|██████████| 585/585 [06:52<00:00,  1.42it/s]

CPU times: user 4min, sys: 38.4 s, total: 4min 38s
Wall time: 6min 52s


### Write ```definition.json```

In [13]:
%%writefile definition.json

{
  "Comment": "A description of my state machine",
  "StartAt": "Map",
  "States": {
    "Map": {
      "Type": "Map",
      "ItemProcessor": {
        "ProcessorConfig": {
          "Mode": "DISTRIBUTED",
          "ExecutionType": "STANDARD"
        },
        "StartAt": "ParsePayloads",
        "States": {
          "ParsePayloads": {
            "Type": "Task",
            "Resource": "arn:aws:states:::lambda:invoke",
            "OutputPath": "$.Payload",
            "Parameters": {
              "Payload.$": "$",
              "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:gen-xii-retro-scoring:$LATEST"
            },
            "Retry": [
              {
                "ErrorEquals": [
                  "Lambda.ServiceException",
                  "Lambda.AWSLambdaException",
                  "Lambda.SdkClientException",
                  "Lambda.TooManyRequestsException"
                ],
                "IntervalSeconds": 1,
                "MaxAttempts": 3,
                "BackoffRate": 2
              }
            ],
            "End": true
          }
        }
      },
      "End": true,
      "Label": "Map",
      "MaxConcurrency": 1000,
      "ItemReader": {
        "Resource": "arn:aws:states:::s3:getObject",
        "ReaderConfig": {
          "InputType": "CSV",
          "CSVHeaderLocation": "FIRST_ROW"
        },
        "Parameters": {
          "Bucket": "20231010-gen-xii",
          "Key": "08_retro_scoring/03_step_function/df_idx.csv"
        }
      }
    }
  }
}

Writing definition.json


### Make string definition

In [14]:
# load it
dict_definition = json.load(open('./definition.json'))
# make into string
str_definition = json.dumps(dict_definition)

### Create state machine

In [15]:
cls_client_sfn = boto3.client('stepfunctions')

In [16]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [17]:
# list state machines
dict_response = cls_client_sfn.list_state_machines(
)
list_dict_state_machines = dict_response['stateMachines']
list_dict_state_names = [{dict_state_machine['name']: dict_state_machine['stateMachineArn']} for dict_state_machine in list_dict_state_machines]
dict_state_names = {key: val for dict_name in list_dict_state_names for key, val in dict_name.items()}
pprint(dict_state_names)

{'MyStateMachine-fldl4s6of': 'arn:aws:states:us-west-2:836690756591:stateMachine:MyStateMachine-fldl4s6of',
 'gen-xi-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xi-retro-scoring',
 'gen-xii-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-retro-scoring',
 'poc-step-genxii-lgd-lambda-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:poc-step-genxii-lgd-lambda-boto3',
 'poc-step-genxii-pd-lambda-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:poc-step-genxii-pd-lambda-boto3',
 'step-genxii-ad-feat-select-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-feat-select-boto3',
 'step-genxii-ad-model-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-model-boto3',
 'step-genxii-ad-pre-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-pre-boto3',
 'step-genxii-lgd-feat-select-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxi

In [18]:
# get list of just names
list_str_names = [list(dict_state_names.keys())[0] for dict_state_names in list_dict_state_names]
# if our name is in there
if str_name in list_str_names:
    print(f'State machine {str_name} exists, it will be deleted')
    str_arn = dict_state_names[str_name]
    print(f'Deleting {str_arn}')
    print('')
    dict_response = cls_client_sfn.delete_state_machine(
        stateMachineArn=str_arn,
    )
    pprint(dict_response)
else:
    print(f'State machine {str_name} does not exist, so it will not be deleted')

State machine gen-xii-retro-scoring exists, it will be deleted
Deleting arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-retro-scoring

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '2',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Thu, 07 Mar 2024 20:58:48 GMT',
                                      'x-amzn-requestid': '3628e9a4-4589-4876-92dd-558859584bf2'},
                      'HTTPStatusCode': 200,
                      'RequestId': '3628e9a4-4589-4876-92dd-558859584bf2',
                      'RetryAttempts': 0}}


In [19]:
# make a state machine
while True:
    try:
        dict_response = cls_client_sfn.create_state_machine(
            name=str_name,
            definition=str_definition,
            roleArn=str_role,
            type='STANDARD',
        )
        pprint(dict_response)
        break
    except:
        pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '126',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Thu, 07 Mar 2024 20:59:51 GMT',
                                      'x-amzn-requestid': '0dc39343-ffbe-4d7d-ab85-b111f6ba19bc'},
                      'HTTPStatusCode': 200,
                      'RequestId': '0dc39343-ffbe-4d7d-ab85-b111f6ba19bc',
                      'RetryAttempts': 3},
 'creationDate': datetime.datetime(2024, 3, 7, 20, 59, 51, 951000, tzinfo=tzlocal()),
 'stateMachineArn': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-retro-scoring'}


### Describe state machine

In [20]:
str_state_machine_arn = dict_response['stateMachineArn']
print(f'State Machine ARN: {str_state_machine_arn}')
dict_response = cls_client_sfn.describe_state_machine(
    stateMachineArn=str_state_machine_arn,
)
pprint(dict_response)

State Machine ARN: arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-retro-scoring
{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1589',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Thu, 07 Mar 2024 20:59:52 GMT',
                                      'x-amzn-requestid': '0aac96fc-51d0-400f-beb1-9a76f641e518'},
                      'HTTPStatusCode': 200,
                      'RequestId': '0aac96fc-51d0-400f-beb1-9a76f641e518',
                      'RetryAttempts': 0},
 'creationDate': datetime.datetime(2024, 3, 7, 20, 59, 51, 951000, tzinfo=tzlocal()),
 'definition': '{"Comment": "A description of my state machine", "StartAt": '
               '"Map", "States": {"Map": {"Type": "Map", "ItemProcessor": '
               '{"ProcessorConfig": {"Mode": "DISTRIBUTED", "ExecutionType": '
               '"STANDARD"}, "Star

### Execute step function workflow

In [21]:
# # start execution
# dict_response = cls_client_sfn.start_execution(
#     stateMachineArn=str_state_machine_arn,
# )

### Clean-up

In [22]:
os.remove('./definition.json')